## Подключение

Ячейка подключения выполняется первой в каждом сеансе. Она подключает Google Drive (личная папка `practice2026` и общие данные группы `kg_data`), клонирует репозиторий группы в `/content/kg-practice-2026`, переключается на свою ветку `student-NN` с работой группы из `main` и регистрирует модули группы. В первых строках студент указывает свой номер, модуль и фамилию латиницей; токен GitHub лежит в секретах Colab под именем `GITHUB_TOKEN` (`ИНСТРУКЦИЯ_git.md`). Сдача этапа — последняя ячейка блокнота: `repo.submit(...)`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import sys, os, json, time, subprocess

STUDENT = 'student-07'                     # ← свой номер: вариант 7 — student-07
MY_MODULE = 'm07'                          # ← свой модуль: вариант 7 — m07
SURNAME = 'ivanov'                         # ← фамилия латиницей строчными буквами
REPO = 'github.com/<организация>/kg-practice-2026'     # репозиторий группы

KG = '/content/kg-practice-2026'                       # клон репозитория: код, модули, эталоны, сдача
DATA = '/content/drive/MyDrive/kg_data'                # общие данные группы на Drive (только чтение)
WORK = '/content/drive/MyDrive/practice2026'           # личная папка на Drive: свой набор и результаты
TEST = DATA + '/test_set'                              # общий тестовый набор
TEST_FACTS = WORK + '/test_facts'                      # свои прогоны модулей на тестовом наборе
ONTOLOGY = [KG + '/ontology/MocassinPlus_v2.ttl', KG + '/ontology/SciLibReco.ttl']
os.makedirs(WORK, exist_ok=True)

TOKEN = userdata.get('GITHUB_TOKEN')                   # секрет Colab с токеном GitHub
if not os.path.exists(KG):
    r = subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{REPO}.git', KG], capture_output=True, text=True)
    print(r.stderr.replace(TOKEN, '***'))
sys.path.insert(0, KG)
!pip install -q -r {KG}/requirements.txt
from mathkg.gitwork import Repo
repo = Repo(KG, STUDENT, MY_MODULE, SURNAME, token=TOKEN)
repo.start()                                           # своя ветка и всё, что уже принято в main
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
print('свой модуль:', MY_MODULE, my_tool)

# Этап 04. Модуль: модель Hugging Face

Лекция 4 · сдать к встрече 6 · работа студента: 6–8 ч · вычисления в Colab (GPU T4): подготовка
обучающей выборки на своём наборе — 5–15 мин; кодирование `multilingual-e5-base` 20–50 тыс. абзацев
— 3–8 мин; обучение логистической регрессии — секунды; zero-shot `mDeBERTa` на кандидатах тестового
набора — 1–3 мин.

## Цель

Студент добавляет в свой модуль модель Hugging Face для случаев, которые правила пропускают:
определение без заголовка «Определение», утверждение без слова «Теорема», конец доказательства без
знака □, имя автора в нестандартной строке, тип раздела с необычным названием. Модель работает
после правил и только на кандидатах, которые правила не взяли; её находки помечаются
`method="model"` с уверенностью модели. Студент сравнивает два подхода — zero-shot классификацию и
логистическую регрессию на эмбеддингах, обученную на серебряной разметке из своих правил, — и
оставляет лучший по F1 на эталоне.

## Что прочитать

* SLP3, гл. 4 «Logistic Regression»: признаки, сигмоида и softmax, перекрёстная энтропия,
  регуляризация, оценка.
* SLP3, гл. 9 «Masked Language Models»: кодировщики и их использование для классификации.
* Alammar, Grootendorst, гл. 4 «Text Classification»: готовая модель, эмбеддинги с классификатором,
  zero-shot.
* Alammar, Grootendorst, гл. 11, разд. «Supervised Classification», «Few-Shot Classification».

## Вход

* `modules/mNN_*.py` — модуль на правилах после этапа 3, `gold/mNN.jsonl`.
* `practice2026/base/` — свой набор (источник серебряной разметки).
* Тестовый набор и `practice2026/test_facts/`.

## Шаги

### 1. Что классифицировать

Модель решает задачу классификации одной единицы текста. Единица и классы зависят от модуля:

| Модуль | Единица | Классы |
|---|---|---|
| m02 Структура | короткий абзац-заголовок | тип раздела: введение, предварительные сведения, результаты, заключение, приложение, прочее |
| m03 Утверждения | абзац без заголовка блока | утверждение / не утверждение |
| m04 Определения | предложение | определение / не определение |
| m07 Доказательства | абзац после начала доказательства | продолжение доказательства / новый текст |
| m10 Предметные области | аннотация | раздел arXiv |
| m01, m06, m09 | строка, запись, словосочетание | NER (шаг 5) |
| m05, m08 | формула или ссылка | см. вариант |

Кандидаты отбираются дёшево: единицы, которые правила не покрыли, с опорными словами
(«называется», «is called», «пусть», «тогда»). Это ограничивает число вызовов модели и удерживает
время модуля в пределах 2 с на статью.

Код этапа написан на примере модуля 4 (m04): единица — предложение, опорные слова — обороты
определений. Студент другого модуля задаёт в ячейке ниже свою единицу (`UNIT`), свои опорные слова
(`CUE`) и свои метки zero-shot (`LABELS`, `TARGET`) из раздела «Этап 4» своего варианта; остальной код
блокнота от модуля не зависит.

In [ ]:
import re, random
import numpy as np, pandas as pd
from mathkg.base import read_doc
from mathkg.pipeline import load_facts

UNIT = 'sentences'          # единица классификации: 'sentences' или 'paragraphs'
CUE = re.compile(r'\b(называ\w+|назов[её]м|будем называть|означает|определ\w+|is called|are called|'
                 r'we say|is said to be|we call|is defined|denote[sd]?|is referred to as)\b', re.I)
LABELS = ['определение понятия', 'формулировка теоремы', 'доказательство', 'пример', 'другое']
TARGET = 'определение понятия'                 # метка, которая означает «свой класс»

def units(d):
    return d.sentences if UNIT == 'sentences' else d.paragraphs

def candidates(d, facts):
    """Единицы статьи с опорными словами, которые правила модуля не покрыли."""
    covered = [(f.start, f.end) for f in (facts.fragments if facts else [])]
    return [u for u in units(d)
            if not any(a <= u.start < b for a, b in covered) and CUE.search(d.text[u.start:u.end])]

test_ids = pd.read_json(TEST + '/base/docs.jsonl', lines=True).query("status == 'ok'").doc_id.tolist()
pipeline.run_corpus(TEST + '/base', TEST_FACTS, names=[my_tool])     # правила своего модуля
cand = []
for doc_id in test_ids:
    d = read_doc(TEST + '/base', doc_id)
    cand += [(doc_id, ' '.join(d.text[u.start:u.end].split())[:600])
             for u in candidates(d, load_facts(TEST_FACTS, my_tool, doc_id))]
print(len(cand), 'кандидатов на', len(test_ids), 'статьях тестового набора')
for doc_id, t in cand[:5]:
    print(doc_id, '|', t[:150])

### 2. Zero-shot классификация

Модель вывода (NLI) проверяет, следует ли из текста гипотеза «Этот текст — определение понятия».
Обучение не нужно; классы задаются словами. Модель берётся из общего кэша `mathkg.models`: она
загружается один раз за сеанс, и модуль, которому она понадобится позже, получит ту же копию.

In [ ]:
from mathkg.models import get_pipeline
zs = get_pipeline('zero-shot-classification', 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli')

text = 'Пространство X будем называть равномерно выпуклым, если для любого ε > 0 найдётся δ > 0 …'
r = zs(text, candidate_labels=LABELS, hypothesis_template='Этот текст — {}.')
print(list(zip(r['labels'], [round(s, 2) for s in r['scores']])))

Для английских статей метки и шаблон даются по-английски (`'This text is a {}.'`). Ячейка ниже
прогоняет zero-shot по 20 кандидатам тестового набора; студент пробует две-три формулировки меток и
записывает в отчёт, какая работает лучше.

In [ ]:
LABELS_EN = ['definition of a concept', 'statement of a theorem', 'proof', 'example', 'other']
rows = []
for doc_id, t in cand[:20]:
    en = not re.search('[А-Яа-яЁё]', t)
    r = zs(t, candidate_labels=LABELS_EN if en else LABELS,
           hypothesis_template='This text is a {}.' if en else 'Этот текст — {}.')
    top = LABELS[(LABELS_EN if en else LABELS).index(r['labels'][0])]
    rows.append(dict(doc_id=doc_id, label=top, score=round(r['scores'][0], 2), text=t[:90]))
zs_table = pd.DataFrame(rows)
zs_table

### 3. Эмбеддинги и логистическая регрессия на серебряной разметке

Правила с высокой уверенностью (заголовок «Определение 1.» полужирным) дают на своём наборе из
1000 статей тысячи надёжных примеров. Их тексты — положительные примеры; единицы из абзацев, которые
правила не взяли, — отрицательные. Такая разметка называется серебряной: она получена автоматически
и содержит ошибки, но её много.

У серебряной разметки модуля, который находит блоки по заголовку, есть ловушка. Каждый
положительный пример начинается со слова «Определение» или «Definition», и классификатор выучивает
это слово вместо содержания. На кросс-валидации такой классификатор выглядит отлично, а на
предложениях без заголовка, ради которых модель и добавляется, ошибается. В эксперименте к лекции 4
(21 статья, 253 абзаца) снятие заголовков с обучающих примеров снизило F1 на серебряной разметке с
0,89 до 0,68 и подняло число верных ответов на шести абзацах без заголовка с трёх до пяти; zero-shot
на тех же абзацах дал два верных ответа. Поэтому заголовок блока (`HEADER`) с положительных примеров
снимается. Модулям, у которых правило не опирается на заголовок (структура, предметные области),
снятие не нужно: `HEADER` для них оставляют пустым шаблоном `r'(?!)'`.

In [ ]:
pipeline.setup(student_modules=KG + '/modules')
my_ids = pd.read_json(WORK + '/base/docs.jsonl', lines=True).query("status == 'ok'").doc_id.tolist()
sample = random.Random(0).sample(my_ids, min(300, len(my_ids)))
pipeline.run_corpus(WORK + '/base', WORK + '/facts', names=[my_tool], doc_ids=sample)

HEADER = re.compile(r'^\s*(Определение|Теорема|Лемма|Утверждение|Предложение|Следствие|Замечание|Пример|'
                    r'Доказательство|Definition|Theorem|Lemma|Proposition|Corollary|Claim|Conjecture|'
                    r'Remark|Example|Proof)\b[^.:]{0,60}[.:]\s*', re.I)
pos, neg = [], []
for doc_id in sample:
    d = read_doc(WORK + '/base', doc_id)
    f = load_facts(WORK + '/facts', my_tool, doc_id)
    if f is None:
        continue
    covered = [(x.start, x.end) for x in f.fragments]
    for x in f.fragments:
        if x.confidence >= 0.9:                         # только надёжные находки правил
            t = HEADER.sub('', ' '.join(d.text[x.start:x.end].split()), count=1)[:600]
            if len(t) > 40:
                pos.append(t)
    free = [u for u in units(d) if not any(a < u.end and u.start < b for a, b in covered)
            and 60 < u.end - u.start < 1500]
    for u in random.Random(doc_id).sample(free, min(8, len(free))):
        t = ' '.join(d.text[u.start:u.end].split())[:600]
        if not HEADER.match(t):
            neg.append(t)
print(len(pos), 'положительных,', len(neg), 'отрицательных')

In [ ]:
from mathkg.models import get_sentence_model
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import joblib

enc = get_sentence_model('intfloat/multilingual-e5-base')   # GPU, если он есть; иначе процессор
X = enc.encode(['passage: ' + t for t in pos + neg], batch_size=64, normalize_embeddings=True,
               show_progress_bar=True)
y = np.array([1] * len(pos) + [0] * len(neg))
clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0)
print('F1 на кросс-валидации:', cross_val_score(clf, X, y, cv=5, scoring='f1').mean().round(3))
clf.fit(X, y)
joblib.dump(clf, f'{KG}/modules/{MY_MODULE}_clf.joblib')

Кросс-валидация на серебряной разметке показывает только, насколько классификатор воспроизводит
правила. Настоящая проверка — эталон (шаг 6). Перед ней полезно посмотреть, что классификатор
говорит о кандидатах тестового набора, рядом с ответами zero-shot:

In [ ]:
p = clf.predict_proba(enc.encode(['passage: ' + t for _, t in cand[:20]], normalize_embeddings=True))[:, 1]
zs_table['p_logreg'] = p.round(2)
zs_table[['doc_id', 'label', 'score', 'p_logreg', 'text']]

### 4. Модель внутри модуля

Модель загружается один раз при первом вызове модуля: кодировщик берётся из общего кэша
`mathkg.models`, классификатор читается из файла рядом с модулем и хранится в переменной модуля.
Ступень модуля задаётся переменной окружения `MATHKG_MNN` (`rules`, `model` или `llm`; по умолчанию
`model`), порог уверенности — переменной `MATHKG_MNN_THR`. Так на шаге 6 порог подбирается без правки
кода, на этапе 6 строится таблица прироста по ступеням, а на этапе 7 прогон набора идёт без языковой
модели.

Готовый образец для модуля 4 (m04) лежит в `mathkg/modules_template/m04_model_example.py`:
ступень `rules` вызывает базовую версию модуля, ступень `model` добавляет находки классификатора и
заново строит связи `moc:isPartOf` и `moc:defines`. Студент модуля 4 копирует его в
`modules/m04_definitions.py` (если своего файла модуля ещё нет) или переносит из него
функцию `model_candidates` в свой файл. Главная часть образца:

```py
# фрагмент файла modules/m04_definitions.py (в блокноте не выполняется)
import os
LEVEL = os.environ.get('MATHKG_M04', 'model')                  # rules | model | llm
THRESHOLD = float(os.environ.get('MATHKG_M04_THR', '0.8'))
CLF_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'm04_clf.joblib')
_clf = None

def _classifier():
    """Классификатор загружается один раз за сеанс; без файла ступень model совпадает с rules."""
    global _clf
    if _clf is None and os.path.exists(CLF_PATH):
        import joblib
        _clf = joblib.load(CLF_PATH)
    return _clf

def model_candidates(doc, facts, threshold=THRESHOLD):
    """Предложения с опорными словами, не покрытые правилами, -> фрагменты method='model'."""
    clf = _classifier()
    covered = [(f.start, f.end) for f in facts.fragments]
    cand = [s for s in doc.sentences
            if not any(a <= s.start < b for a, b in covered) and CUE.search(doc.text[s.start:s.end])]
    if clf is None or not cand:
        return 0
    from mathkg.models import get_sentence_model
    enc = get_sentence_model('intfloat/multilingual-e5-base')
    X = enc.encode(['passage: ' + doc.text[s.start:s.end] for s in cand], batch_size=32,
                   normalize_embeddings=True, show_progress_bar=False)
    found = 0
    for s, p in zip(cand, clf.predict_proba(X)[:, 1]):
        if p >= threshold:
            facts.add_fragment(id='def-tmp', cls='moc:Definition', start=s.start, end=s.end,
                               page=s.page, method='model', confidence=round(float(p), 3))
            found += 1
    return found
```

В функции модуля ступень модели вызывается после правил; если модель что-то нашла, фрагменты
перенумеровываются (`numbered` из `mathkg.modules._util`), и связи фрагментов строятся заново.
Студент модуля 4 копирует образец в папку `modules/` своего клона так (студенты других модулей эту
ячейку пропускают: после слияния в `main` файл в `modules/` заменяет модуль у всей группы):

In [ ]:
import shutil
if MY_MODULE == 'm04' and not os.path.exists(KG + '/modules/m04_definitions.py'):
    shutil.copy(KG + '/mathkg/modules_template/m04_model_example.py', KG + '/modules/m04_definitions.py')
pipeline.setup(student_modules=KG + '/modules')
print(registry.REGISTRY[my_tool].author or 'базовая версия')

Время: кодирование 20–50 предложений на GPU занимает десятые доли секунды. На CPU (прогон других
студентов без GPU) — до нескольких секунд; поэтому отбор кандидатов должен быть узким.

### 5. NER для модулей 1, 6, 9

Модулям метаданных, литературы и понятий вместо классификатора подходит модель NER:
`Babelscape/wikineural-multilingual-ner` (персоны, организации) или Natasha для русского текста.
Модель проверяет кандидатов правил (имя в строке авторов, фамилия в записи литературы) и находит
пропущенные; уверенность берётся из `score` модели. Модель берётся из того же общего кэша.

In [ ]:
from mathkg.models import get_pipeline

def ner(text):
    m = get_pipeline('token-classification', 'Babelscape/wikineural-multilingual-ner',
                     aggregation_strategy='simple')
    return [e for e in m(text) if e['entity_group'] in ('PER', 'ORG')]

if MY_MODULE in ('m01', 'm06', 'm09'):
    print(ner('Ivanov I. I., Lomonosov Moscow State University, Moscow, Russia'))

### 6. Сравнение на эталоне и выбор порога

Порог уверенности модели выбирается на пяти статьях эталона, итоговое качество сообщается на других
пяти, чтобы не подгонять порог под проверочные данные. Эталон делится по статьям на два файла: на
половине «выбор» перебираются пороги, на половине «отчёт» сравниваются ступени `rules` и `model` с
выбранным порогом. Функция `evaluate` считает только статьи из переданного ей файла эталона, поэтому
модуль прогоняется на всём тестовом наборе.

In [ ]:
from mathkg.evaluate import evaluate

gold = [json.loads(l) for l in open(f'{KG}/gold/{MY_MODULE}.jsonl', encoding='utf-8') if l.strip()]
gold_ids = sorted({g['doc_id'] for g in gold})
os.makedirs(WORK + '/stage04', exist_ok=True)

def gold_part(ids, name):
    path = f'{WORK}/stage04/gold_{name}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for g in gold:
            if g['doc_id'] in ids:
                f.write(json.dumps(g, ensure_ascii=False) + '\n')
    return path

gold_select, gold_report = gold_part(gold_ids[::2], 'select'), gold_part(gold_ids[1::2], 'report')
from collections import Counter
KEY = Counter(g['cls'] for g in gold if 'cls' in g).most_common(1)[0][0]   # основной класс своего эталона
print('класс для выбора порога:', KEY)

def run(level, thr=0.8):
    """Прогон своего модуля на тестовом наборе с заданной ступенью и порогом."""
    os.environ[f'MATHKG_{MY_MODULE.upper()}'] = level
    os.environ[f'MATHKG_{MY_MODULE.upper()}_THR'] = str(thr)
    pipeline.setup(student_modules=KG + '/modules')      # перечитывает модуль с новыми настройками
    pipeline.run_corpus(TEST + '/base', TEST_FACTS, names=[my_tool], redo=[my_tool])

rows = []
for thr in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    run('model', thr)
    rows.append(dict(threshold=thr, **evaluate(my_tool, gold_select, TEST + '/base', TEST_FACTS)['by_key'][KEY]))
by_thr = pd.DataFrame(rows)
best_thr = float(by_thr.sort_values(['f1', 'threshold'], ascending=[False, False]).iloc[0].threshold)
print('выбранный порог:', best_thr)
by_thr

In [ ]:
rows = []
for level in ['rules', 'model']:
    run(level, best_thr)
    r = evaluate(my_tool, gold_report, TEST + '/base', TEST_FACTS)
    for k, v in r['by_key'].items():
        rows.append(dict(level=level, key=k, **v))
report = pd.DataFrame(rows).pivot(index='key', columns='level', values='f1')
report

Если на половине «выбор» несколько порогов дают одинаковый F1, берётся больший: модель с ним добавляет
меньше находок и реже ошибается на чужих статьях. Студент строит такую же таблицу для zero-shot (в
`model_candidates` вместо классификатора — `zs` с меткой `TARGET`) и оставляет в модуле лучший
вариант. Если модель не прибавляет F1, в отчёте объясняется почему (мало кандидатов, серебряная
разметка однобока, классы неразделимы по эмбеддингу) и модель всё равно остаётся ступенью модуля с
порогом, при котором она не ухудшает результат. Находки модели на половине «отчёт» студент
просматривает глазами:

In [ ]:
os.environ[f'MATHKG_{MY_MODULE.upper()}_THR'] = str(best_thr)
run('model', best_thr)
for doc_id in gold_ids[1::2]:
    d = read_doc(TEST + '/base', doc_id)
    for x in load_facts(TEST_FACTS, my_tool, doc_id).fragments:
        if x.method == 'model':
            print(doc_id, x.confidence, '|', ' '.join(d.text[x.start:x.end].split())[:150])

### 7. Проверка контракта

In [ ]:
from mathkg.check import check_module
run('model', best_thr)
rep = check_module(my_tool, TEST + '/base', TEST_FACTS, ONTOLOGY, limit_ms=2000)
print(rep['passed'], rep['ms_per_doc'], rep['errors'][:5])

После выбора порога студент записывает его значением по умолчанию в свой модуль
(`MATHKG_M04_THR`, `'0.8'` → выбранное число), чтобы сборка группы шла с ним.

## Выход

* модуль с двумя ступенями (`rules`, `model`) в `modules/mNN_*.py`;
* файл классификатора `modules/mNN_clf.joblib` (если используется);
* `practice2026/stage04/REPORT.md` (ячейка сдачи копирует его в `students/NN_фамилия/stage04/`).

## Самопроверка

* Модель загружается один раз за сеанс: повторный прогон тестового набора не печатает загрузку
  модели.
* Среднее время модуля на статью на GPU — меньше 2 с; на CPU измерено и записано в отчёт.
* F1 ступени `model` не ниже ступени `rules`.
* Находки модели имеют `method="model"` и уверенность, которую выдала модель для этого кандидата.

## Что сдать

Сдача — ячейка сдачи в конце блокнота и запрос на слияние «Этап 4: фамилия» в `main`
(`ИНСТРУКЦИЯ_git.md`, раздел 4). В репозиторий попадают модуль и файл классификатора (`modules/`), блокнот
`students/NN_фамилия/stage04.ipynb` с выводами и папка `students/NN_фамилия/stage04/` с отчётом
`REPORT.md` и небольшими результатами этапа. В отчёте: что
классифицируется и как отбираются кандидаты, размер серебряной разметки, результаты zero-shot и
логистической регрессии, выбранный порог, таблица F1 по ступеням `rules` и `model`, время на статью,
пять примеров находок модели (верных и неверных).

## Как результат станет инструментом агента

Уверенность находки — сведения, которыми агент пользуется при ответе: факт с уверенностью 0,95 из
правила и факт с уверенностью 0,62 из модели агент передаёт с разной оговоркой. Описание инструмента
(docstring) дополняется строкой о том, какие находки сделаны моделью.

## Частые ошибки

* Модель загружается внутри функции модуля при каждом вызове: прогон 1000 статей длится часами.
* Модель применена ко всем абзацам статьи без отбора кандидатов.
* Заголовок блока («Определение 1.», «Theorem 2») оставлен в положительных примерах: классификатор
  узнаёт заголовок, на кросс-валидации F1 высокий, а предложения без заголовка он пропускает.
* Порог выбран на тех же статьях, на которых сообщается итоговое качество.
* Префикс `passage: ` для e5 пропущен при обучении или при применении: эмбеддинги несопоставимы.
* Файл классификатора сохранён в личную папку, и у других студентов модуль не загружается.

## Вопросы к защите

1. Как работает zero-shot классификация через модель вывода (NLI)?
2. Что такое серебряная разметка и какие систематические ошибки она переносит в классификатор?
3. Почему кросс-валидация на серебряной разметке не оценивает качество модуля?
4. Зачем логистической регрессии `class_weight='balanced'`?
5. Как выбирался порог уверенности и почему не на всех статьях эталона?
6. Как модуль укладывается в 2 с на статью и что будет при прогоне на CPU?

**Вариант.** Единица классификации, классы, опорные слова кандидатов и рекомендуемая модель для
своего модуля указаны в разделе «Этап 4» файла `варианты/Вариант_NN.md`.

## Сдача этапа

Перед сдачей блокнот сохраняют (Ctrl+S), а отчёт `REPORT.md` кладут в папку этапа на Drive (`practice2026/stageNN/`). Ячейка копирует в свою папку сдачи `students/NN_фамилия/` блокнот и небольшие файлы этапа (отчёт, таблицы, запросы, журналы), отправляет свои файлы в ветку `student-NN` и печатает ссылку на запрос на слияние в `main`; запрос открывают по ссылке с заголовком «Этап N: фамилия».

In [ ]:
STAGE = 4
repo.save_notebook(f'stage{STAGE:02d}_{SURNAME}.ipynb', STAGE)   # имя своей копии блокнота на Drive
repo.collect(f'{WORK}/stage{STAGE:02d}', STAGE)                  # отчёт и небольшие результаты этапа
repo.submit(f'Этап {STAGE}: {SURNAME}')